In [1]:
1+1

2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py

from scipy.sparse import csc_matrix
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

In [3]:
# Repository root
root = Path.cwd().parent

raw_dir = root / "data" / "raw"
figures_dir = root / "figures"
tables_dir = root / "tables"

figures_dir.mkdir(exist_ok=True)
tables_dir.mkdir(exist_ok=True)

print("Root:", root)
print("Raw data:", raw_dir)
print("Figures:", figures_dir)
print("Tables:", tables_dir)

Root: c:\Users\shafi\number-simplex-reproduction
Raw data: c:\Users\shafi\number-simplex-reproduction\data\raw
Figures: c:\Users\shafi\number-simplex-reproduction\figures
Tables: c:\Users\shafi\number-simplex-reproduction\tables


In [4]:
print([
    p.name
    for p in raw_dir.iterdir()
    if p.is_dir()
])

['YFF', 'YFI', 'YFJ', 'YFK', 'YFL', 'YFM', 'YFP', 'YFR', 'YFS', 'YFT', 'YFU']


In [5]:
subject = "YFU"

yfu_dir = raw_dir / subject / "arithmetic"

spike_path = yfu_dir / "spikes.mat"
behav_path = yfu_dir / "photoBehavEvents.csv"

print("Spike file exists:", spike_path.exists())
print("Behavior file exists:", behav_path.exists())

Spike file exists: True
Behavior file exists: True


In [6]:
def decode_matlab_string(f, ref):
    obj = f[ref]
    values = obj[()].flatten()
    return "".join(chr(int(x)) for x in values)

In [7]:
with h5py.File(spike_path, "r") as f:

    spikes = f["spikes"]

    yfu_spike_matrix = csc_matrix(
        (
            spikes["data"][:],
            spikes["ir"][:],
            spikes["jc"][:]
        ),
        shape=(
            int(spikes.attrs["MATLAB_sparse"]),
            len(spikes["jc"]) - 1
        )
    )

    regions_obj = f["regionsVect"]

    yfu_region_labels = [
        decode_matlab_string(f, ref)
        for ref in regions_obj[0]
    ]

In [8]:
yfu_behav = pd.read_csv(behav_path)

In [9]:
print("YFU spike matrix:", yfu_spike_matrix.shape)
print("YFU behavior:", yfu_behav.shape)
print("Region labels:", len(yfu_region_labels))

unique_regions, counts = np.unique(
    yfu_region_labels,
    return_counts=True
)

print("\nRegions:")
for region, count in zip(unique_regions, counts):
    print(region, count)

YFU spike matrix: (78, 2273935)
YFU behavior: (300, 22)
Region labels: 78

Regions:
acc 16
amy 23
hpc 39


In [10]:
yfu_aligned = yfu_behav.copy()

yfu_aligned["operand1_time"] = np.where(
    yfu_aligned["operationFirst"] == 1,
    yfu_aligned["tCue2"],
    yfu_aligned["tCue1"]
)

yfu_aligned["operation_time"] = np.where(
    yfu_aligned["operationFirst"] == 1,
    yfu_aligned["tCue1"],
    yfu_aligned["tCue3"]
)

yfu_aligned["operand2_time"] = np.where(
    yfu_aligned["operationFirst"] == 1,
    yfu_aligned["tCue3"],
    yfu_aligned["tCue2"]
)

In [11]:
display(
    yfu_aligned[
        [
            "cue1",
            "operation",
            "cue2",
            "operationFirst",
            "tCue1",
            "tCue2",
            "tCue3",
            "operand1_time",
            "operation_time",
            "operand2_time"
        ]
    ].head(10)
)

,cue1,operation,cue2,operationFirst,tCue1,tCue2,tCue3,operand1_time,operation_time,operand2_time
0,7,-,8,1,5910.833333,6660.833333,7410.800000,6660.833333,5910.833333,7410.800000
1,7,+,4,0,81376.233333,82126.233333,82876.233333,81376.233333,82876.233333,82126.233333
2,7,-,6,1,88876.133333,89626.100000,90376.100000,89626.100000,88876.133333,90376.100000
3,10,-,5,1,96092.666667,96842.633333,97592.633333,96842.633333,96092.666667,97592.633333
4,8,-,0,0,118708.933333,119458.933333,120208.933333,118708.933333,120208.933333,119458.933333
5,4,+,4,1,123892.200000,124642.166667,125392.166667,124642.166667,123892.200000,125392.166667
6,10,+,2,1,129525.433333,130275.433333,131025.400000,130275.433333,129525.433333,131025.400000
7,7,-,3,1,135025.333333,135775.333333,136525.300000,135775.333333,135025.333333,136525.300000
8,3,-,2,0,143491.866667,144241.833333,144991.833333,143491.866667,144991.833333,144241.833333
9,4,+,4,0,148841.766667,149575.100000,150341.733333,148841.766667,150341.733333,149575.100000


In [12]:
paper_unit = 43
neuron_id = paper_unit - 1

print("Python row:", neuron_id)
print("Region:", yfu_region_labels[neuron_id])

Python row: 42
Region: hpc


In [13]:
neuron_id = 42

window_start_ms = 50
window_end_ms = 950
window_sec = 0.9

X_fr_rows = []
y_fr_labels = []

for _, row in yfu_aligned.iterrows():

    # Operand 1
    number1 = row["cue1"]
    onset1 = row["operand1_time"]

    if (
        pd.notna(number1)
        and pd.notna(onset1)
        and 1 <= number1 <= 9
    ):
        start = int(round(onset1 + window_start_ms))
        end = int(round(onset1 + window_end_ms))

        spike_count = yfu_spike_matrix[
            neuron_id,
            start:end
        ].sum()

        firing_rate_hz = float(spike_count) / window_sec

        X_fr_rows.append([firing_rate_hz])
        y_fr_labels.append(int(number1))

    # Operand 2
    number2 = row["cue2"]
    onset2 = row["operand2_time"]

    if (
        pd.notna(number2)
        and pd.notna(onset2)
        and 1 <= number2 <= 9
    ):
        start = int(round(onset2 + window_start_ms))
        end = int(round(onset2 + window_end_ms))

        spike_count = yfu_spike_matrix[
            neuron_id,
            start:end
        ].sum()

        firing_rate_hz = float(spike_count) / window_sec

        X_fr_rows.append([firing_rate_hz])
        y_fr_labels.append(int(number2))

In [14]:
X_fr = np.asarray(X_fr_rows, dtype=float)
y_fr = np.asarray(y_fr_labels)

print("X_fr shape:", X_fr.shape)
print("y_fr shape:", y_fr.shape)

print("\nFirst 10 firing rates:")
print(X_fr[:10].ravel())

print("\nFirst 10 labels:")
print(y_fr[:10])

X_fr shape: (494, 1)
y_fr shape: (494,)

First 10 firing rates:
[ 6.66666667  7.77777778  6.66666667  7.77777778  7.77777778  6.66666667
  4.44444444  4.44444444  3.33333333 10.        ]

First 10 labels:
[7 8 7 4 7 6 5 8 4 4]


In [15]:
unique_numbers, counts = np.unique(
    y_fr,
    return_counts=True
)

for number, count in zip(unique_numbers, counts):
    print(f"Number {number}: {count} presentations")

Number 1: 53 presentations
Number 2: 67 presentations
Number 3: 52 presentations
Number 4: 62 presentations
Number 5: 51 presentations
Number 6: 48 presentations
Number 7: 65 presentations
Number 8: 49 presentations
Number 9: 47 presentations


In [16]:
def cv_fr_accuracy(X, y, random_state=42):

    classes, counts = np.unique(
        y,
        return_counts=True
    )

    # Use up to 10 folds
    n_folds = min(10, counts.min())

    cv = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=random_state
    )

    # Equal prior probability for numbers 1–9
    priors = np.ones(len(classes)) / len(classes)

    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        lda = LinearDiscriminantAnalysis(
            priors=priors
        )

        lda.fit(X_train, y_train)

        y_pred = lda.predict(X_test)

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

    accuracy = accuracy_score(
        y_true_all,
        y_pred_all
    )

    return accuracy, n_folds

In [17]:
fr_accuracy, n_folds = cv_fr_accuracy(
    X_fr,
    y_fr
)

print("CV folds:", n_folds)
print(
    f"FR decoding accuracy: "
    f"{fr_accuracy * 100:.2f}%"
)

print(
    f"Chance level: "
    f"{100/9:.2f}%"
)

CV folds: 10
FR decoding accuracy: 11.13%
Chance level: 11.11%


In [18]:
# ============================================================
# STEP 6 — Permutation test for FR decoding
# YFU.hpc.43
# ============================================================

n_shuffles = 200
rng = np.random.default_rng(42)

shuffle_accuracies = []

for i in range(n_shuffles):

    # Randomly break the relationship
    # between firing rate and number identity
    y_shuffled = rng.permutation(y_fr)

    shuffled_accuracy, _ = cv_fr_accuracy(
        X_fr,
        y_shuffled,
        random_state=42
    )

    shuffle_accuracies.append(
        shuffled_accuracy
    )

shuffle_accuracies = np.asarray(
    shuffle_accuracies
)

In [19]:
n_extreme = np.sum(
    shuffle_accuracies >= fr_accuracy
)

p_value = (
    1 + n_extreme
) / (
    n_shuffles + 1
)

print(
    f"Observed FR accuracy: "
    f"{fr_accuracy * 100:.2f}%"
)

print(
    f"Mean shuffled accuracy: "
    f"{shuffle_accuracies.mean() * 100:.2f}%"
)

print(
    f"Shuffles >= observed: "
    f"{n_extreme}/{n_shuffles}"
)

print(
    f"Permutation p-value: "
    f"{p_value:.4f}"
)

print()

if p_value < 0.05:
    print("SIGNIFICANT FR number-coding neuron")
else:
    print("NOT significant FR number-coding neuron")

Observed FR accuracy: 11.13%
Mean shuffled accuracy: 10.97%
Shuffles >= observed: 95/200
Permutation p-value: 0.4776

NOT significant FR number-coding neuron


---

Singel usable function for firing rate decoding 

In [20]:
def decode_fr_one_neuron(
    spike_matrix,
    presentations,
    neuron_id,
    n_shuffles=200,
    random_state=42
):
    """
    Firing-rate number decoding for one neuron.

    Parameters
    ----------
    spike_matrix : sparse matrix
        neurons x time samples

    presentations : DataFrame
        One row per number presentation.
        Must contain:
            number
            onset_ms

    neuron_id : int
        Python zero-based neuron index.

    Returns
    -------
    dict
    """

    window_start_ms = 50
    window_end_ms = 950
    window_sec = 0.9

    X_rows = []
    labels = []

    # ---------------------------------------------
    # Build firing-rate feature
    # ---------------------------------------------
    for _, row in presentations.iterrows():

        number = int(row["number"])
        onset = row["onset_ms"]

        start = int(round(
            onset + window_start_ms
        ))

        end = int(round(
            onset + window_end_ms
        ))

        spike_count = spike_matrix[
            neuron_id,
            start:end
        ].sum()

        firing_rate = (
            float(spike_count) / window_sec
        )

        X_rows.append([firing_rate])
        labels.append(number)

    X = np.asarray(X_rows, dtype=float)
    y = np.asarray(labels)

    # ---------------------------------------------
    # Real decoding
    # ---------------------------------------------
    observed_accuracy, n_folds = cv_fr_accuracy(
        X,
        y,
        random_state=random_state
    )

    # ---------------------------------------------
    # Permutation test
    # ---------------------------------------------
    rng = np.random.default_rng(random_state)

    shuffled_accuracies = []

    for _ in range(n_shuffles):

        y_shuffled = rng.permutation(y)

        shuffled_accuracy, _ = cv_fr_accuracy(
            X,
            y_shuffled,
            random_state=random_state
        )

        shuffled_accuracies.append(
            shuffled_accuracy
        )

    shuffled_accuracies = np.asarray(
        shuffled_accuracies
    )

    n_extreme = np.sum(
        shuffled_accuracies >= observed_accuracy
    )

    p_value = (
        1 + n_extreme
    ) / (
        n_shuffles + 1
    )

    # ---------------------------------------------
    # Return result
    # ---------------------------------------------
    return {
        "neuron_id": neuron_id,
        "n_presentations": len(y),
        "n_folds": n_folds,

        "fr_accuracy": observed_accuracy,

        "shuffle_mean_accuracy":
            shuffled_accuracies.mean(),

        "n_extreme": int(n_extreme),

        "p_value": p_value,

        "fr_coding":
            bool(p_value < 0.05)
    }

    

In [21]:
presentation_rows = []

for trial_idx, row in yfu_aligned.iterrows():

    # Operand 1
    if (
        pd.notna(row["cue1"])
        and pd.notna(row["operand1_time"])
        and 1 <= row["cue1"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 1,
            "number": int(row["cue1"]),
            "onset_ms": row["operand1_time"]
        })

    # Operand 2
    if (
        pd.notna(row["cue2"])
        and pd.notna(row["operand2_time"])
        and 1 <= row["cue2"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 2,
            "number": int(row["cue2"]),
            "onset_ms": row["operand2_time"]
        })

yfu_presentations = pd.DataFrame(
    presentation_rows
)

print(yfu_presentations.shape)

display(
    yfu_presentations.head()
)

(494, 4)


,trial_index,operand,number,onset_ms
0,0,1,7,6660.833333
1,0,2,8,7410.800000
2,1,1,7,81376.233333
3,1,2,4,82126.233333
4,2,1,7,89626.100000


In [22]:
test_result = decode_fr_one_neuron(
    spike_matrix=yfu_spike_matrix,
    presentations=yfu_presentations,
    neuron_id=42,
    n_shuffles=200,
    random_state=42
)

test_result

{'neuron_id': 42,
 'n_presentations': 494,
 'n_folds': 10,
 'fr_accuracy': 0.11133603238866396,
 'shuffle_mean_accuracy': np.float64(0.10974696356275306),
 'n_extreme': 95,
 'p_value': np.float64(0.47761194029850745),
 'fr_coding': False}

Run FR analysis for all YFU neurons

In [23]:
yfu_fr_results = []

n_neurons = yfu_spike_matrix.shape[0]

for neuron_id in range(n_neurons):

    result = decode_fr_one_neuron(
        spike_matrix=yfu_spike_matrix,
        presentations=yfu_presentations,
        neuron_id=neuron_id,
        n_shuffles=200,
        random_state=42
    )

    result["subject"] = "YFU"
    result["region"] = yfu_region_labels[neuron_id]

    yfu_fr_results.append(result)

    print(
        f"Neuron {neuron_id + 1}/{n_neurons} completed",
        end="\r"
    )

print("\nDone.")

Neuron 78/78 completed
Done.


In [24]:
yfu_fr_df = pd.DataFrame(yfu_fr_results)

display(yfu_fr_df.head())

print("\nShape:", yfu_fr_df.shape)

print(
    "FR coding neurons:",
    yfu_fr_df["fr_coding"].sum()
)

print(
    "FR coding percentage:",
    100 * yfu_fr_df["fr_coding"].mean()
)

,neuron_id,n_presentations,n_folds,fr_accuracy,shuffle_mean_accuracy,n_extreme,p_value,fr_coding,subject,region
0,0,494,10,0.121457,0.110739,49,0.248756,False,YFU,amy
1,1,494,10,0.105263,0.110202,129,0.646766,False,YFU,amy
2,2,494,10,0.080972,0.110496,195,0.975124,False,YFU,amy
3,3,494,10,0.121457,0.110405,61,0.308458,False,YFU,amy
4,4,494,10,0.107287,0.108401,102,0.512438,False,YFU,amy



Shape: (78, 10)
FR coding neurons: 5
FR coding percentage: 6.41025641025641


In [25]:
display(
    yfu_fr_df[
        yfu_fr_df["neuron_id"] == 42
    ]
)

,neuron_id,n_presentations,n_folds,fr_accuracy,shuffle_mean_accuracy,n_extreme,p_value,fr_coding,subject,region
42,42,494,10,0.111336,0.109747,95,0.477612,False,YFU,hpc


In [26]:
display(
    yfu_fr_df[
        yfu_fr_df["fr_coding"]
    ][
        [
            "neuron_id",
            "region",
            "fr_accuracy",
            "p_value"
        ]
    ]
)

,neuron_id,region,fr_accuracy,p_value
21,21,hpc,0.137652,0.049751
22,22,hpc,0.141700,0.034826
30,30,hpc,0.143725,0.014925
48,48,amy,0.145749,0.039801
75,75,acc,0.143725,0.009950


In [27]:
display(
    yfu_fr_df.groupby("region").agg(
        n_neurons=("neuron_id", "count"),
        n_fr_coding=("fr_coding", "sum")
    )
)

,n_neurons,n_fr_coding
region,,
acc,16,1
amy,23,1
hpc,39,3


In [28]:
mtl_regions = ["hpc", "ent", "amy", "phc"]

yfu_mtl_fr_df = yfu_fr_df[
    yfu_fr_df["region"].isin(mtl_regions)
].copy()

print("YFU MTL neurons:", len(yfu_mtl_fr_df))
print(
    "YFU MTL FR-coding neurons:",
    yfu_mtl_fr_df["fr_coding"].sum()
)
print(
    "YFU MTL FR-coding %:",
    100 * yfu_mtl_fr_df["fr_coding"].mean()
)

YFU MTL neurons: 62
YFU MTL FR-coding neurons: 4
YFU MTL FR-coding %: 6.451612903225806


In [29]:
yfu_mtl_fr_df.to_csv(
    tables_dir / "figure1M_YFU_FR_neuron_results.csv",
    index=False
)

In [30]:
yfu_summary = pd.DataFrame({
    "subject": ["YFU"],
    "n_mtl_neurons": [len(yfu_mtl_fr_df)],
    "n_fr_coding": [yfu_mtl_fr_df["fr_coding"].sum()],
    "fr_coding_percent": [
        100 * yfu_mtl_fr_df["fr_coding"].mean()
    ]
})

display(yfu_summary)

yfu_summary.to_csv(
    tables_dir / "figure1M_YFU_FR_summary.csv",
    index=False
)

,subject,n_mtl_neurons,n_fr_coding,fr_coding_percent
0,YFU,62,4,6.451613


In [31]:
subject = "YFF"

yff_dir = raw_dir / subject / "arithmetic"

spike_path = yff_dir / "spikes.mat"
behav_path = yff_dir / "photoBehavEvents.csv"

print("Spike file exists:", spike_path.exists())
print("Behavior file exists:", behav_path.exists())

Spike file exists: True
Behavior file exists: True


In [32]:
with h5py.File(spike_path, "r") as f:

    spikes = f["spikes"]

    yff_spike_matrix = csc_matrix(
        (
            spikes["data"][:],
            spikes["ir"][:],
            spikes["jc"][:]
        ),
        shape=(
            int(spikes.attrs["MATLAB_sparse"]),
            len(spikes["jc"]) - 1
        )
    )

    regions_obj = f["regionsVect"]

    yff_region_labels = [
        decode_matlab_string(f, ref)
        for ref in regions_obj[0]
    ]

yff_behav = pd.read_csv(behav_path)

In [33]:
print("YFF spike matrix:", yff_spike_matrix.shape)
print("YFF behavior:", yff_behav.shape)
print("Region labels:", len(yff_region_labels))

unique_regions, counts = np.unique(
    yff_region_labels,
    return_counts=True
)

print("\nRegions:")
for region, count in zip(unique_regions, counts):
    print(region, count)

YFF spike matrix: (54, 708276)
YFF behavior: (100, 22)
Region labels: 54

Regions:
acc 17
ent 1
hpc 36


In [34]:
yff_aligned = yff_behav.copy()

yff_aligned["operand1_time"] = np.where(
    yff_aligned["operationFirst"] == 1,
    yff_aligned["tCue2"],
    yff_aligned["tCue1"]
)

yff_aligned["operation_time"] = np.where(
    yff_aligned["operationFirst"] == 1,
    yff_aligned["tCue1"],
    yff_aligned["tCue3"]
)

yff_aligned["operand2_time"] = np.where(
    yff_aligned["operationFirst"] == 1,
    yff_aligned["tCue3"],
    yff_aligned["tCue2"]
)

In [35]:
presentation_rows = []

for trial_idx, row in yff_aligned.iterrows():

    # Operand 1
    if (
        pd.notna(row["cue1"])
        and pd.notna(row["operand1_time"])
        and 1 <= row["cue1"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 1,
            "number": int(row["cue1"]),
            "onset_ms": row["operand1_time"]
        })

    # Operand 2
    if (
        pd.notna(row["cue2"])
        and pd.notna(row["operand2_time"])
        and 1 <= row["cue2"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 2,
            "number": int(row["cue2"]),
            "onset_ms": row["operand2_time"]
        })

yff_presentations = pd.DataFrame(presentation_rows)

In [36]:
print("YFF presentations:", yff_presentations.shape)

display(
    yff_presentations.head(10)
)

YFF presentations: (109, 4)


,trial_index,operand,number,onset_ms
0,0,1,5,6089.466667
1,1,1,2,15882.233333
2,1,2,1,16632.933333
3,2,1,6,23189.233333
4,2,2,1,23939.966667
5,4,2,6,37886.700000
6,5,1,9,43108.400000
7,5,2,8,43859.100000
8,6,1,5,49998.333333
9,7,1,8,58206.233333


In [37]:
display(
    yff_presentations.groupby(
        ["number", "operand"]
    ).size().unstack(fill_value=0)
)

print(
    "\nTotal presentations:",
    len(yff_presentations)
)

operand,1,2
number,,
1,4,6
2,4,11
3,5,6
4,2,5
5,6,1
6,10,6
7,9,8
8,5,7
9,10,4



Total presentations: 109


In [38]:
print("Region:", yff_region_labels[9])

yff_test = decode_fr_one_neuron(
    spike_matrix=yff_spike_matrix,
    presentations=yff_presentations,
    neuron_id=9,
    n_shuffles=200,
    random_state=42
)

yff_test

Region: hpc


{'neuron_id': 9,
 'n_presentations': 109,
 'n_folds': np.int64(7),
 'fr_accuracy': 0.1651376146788991,
 'shuffle_mean_accuracy': np.float64(0.10605504587155964),
 'n_extreme': 9,
 'p_value': np.float64(0.04975124378109453),
 'fr_coding': True}

ALL yFF neurons 

In [39]:
yff_fr_results = []

n_neurons = yff_spike_matrix.shape[0]

for neuron_id in range(n_neurons):

    result = decode_fr_one_neuron(
        spike_matrix=yff_spike_matrix,
        presentations=yff_presentations,
        neuron_id=neuron_id,
        n_shuffles=200,
        random_state=42
    )

    result["subject"] = "YFF"
    result["region"] = yff_region_labels[neuron_id]

    yff_fr_results.append(result)

    print(
        f"Neuron {neuron_id + 1}/{n_neurons} completed",
        end="\r"
    )

print("\nDone.")

Neuron 54/54 completed
Done.


In [40]:
yff_fr_df = pd.DataFrame(yff_fr_results)

print("All recorded units:", len(yff_fr_df))
print("All FR-coding units:", yff_fr_df["fr_coding"].sum())

All recorded units: 54
All FR-coding units: 3


In [41]:
mtl_regions = ["hpc", "ent", "amy", "phc"]

yff_mtl_fr_df = yff_fr_df[
    yff_fr_df["region"].isin(mtl_regions)
].copy()

print("YFF MTL neurons:", len(yff_mtl_fr_df))
print(
    "YFF MTL FR-coding neurons:",
    yff_mtl_fr_df["fr_coding"].sum()
)

print(
    "YFF MTL FR-coding %:",
    100 * yff_mtl_fr_df["fr_coding"].mean()
)

YFF MTL neurons: 37
YFF MTL FR-coding neurons: 1
YFF MTL FR-coding %: 2.7027027027027026


# Full neurons

In [42]:
def run_fr_subject(
    subject,
    spike_matrix,
    presentations,
    region_labels,
    n_shuffles=200,
    random_state=42
):

    results = []

    n_neurons = spike_matrix.shape[0]

    for neuron_id in range(n_neurons):

        result = decode_fr_one_neuron(
            spike_matrix=spike_matrix,
            presentations=presentations,
            neuron_id=neuron_id,
            n_shuffles=n_shuffles,
            random_state=random_state
        )

        result["subject"] = subject
        result["region"] = region_labels[neuron_id]

        results.append(result)

        print(
            f"{subject}: {neuron_id + 1}/{n_neurons}",
            end="\r"
        )

    print(f"\n{subject} complete.")

    # All recorded units
    df = pd.DataFrame(results)

    # Paper's MTL regions only
    mtl_regions = ["hpc", "ent", "amy", "phc"]

    mtl_df = df[
        df["region"].isin(mtl_regions)
    ].copy()

    summary = {
        "subject": subject,
        "n_recorded": len(df),
        "n_mtl_neurons": len(mtl_df),
        "n_fr_coding": int(
            mtl_df["fr_coding"].sum()
        ),
        "fr_coding_percent": (
            100 * mtl_df["fr_coding"].mean()
        )
    }

    return df, mtl_df, summary

In [43]:
yff_mtl_fr_df.to_csv(
    tables_dir / "figure1M_YFF_FR_neuron_results.csv",
    index=False
)

yff_summary = pd.DataFrame([{
    "subject": "YFF",
    "n_mtl_neurons": len(yff_mtl_fr_df),
    "n_fr_coding": int(
        yff_mtl_fr_df["fr_coding"].sum()
    ),
    "fr_coding_percent":
        100 * yff_mtl_fr_df["fr_coding"].mean()
}])

yff_summary.to_csv(
    tables_dir / "figure1M_YFF_FR_summary.csv",
    index=False
)

display(yff_summary)

,subject,n_mtl_neurons,n_fr_coding,fr_coding_percent
0,YFF,37,1,2.702703


In [44]:
subject = "YFI"

yfi_dir = raw_dir / subject / "arithmetic"

spike_path = yfi_dir / "spikes.mat"
behav_path = yfi_dir / "photoBehavEvents.csv"

print("Spike file exists:", spike_path.exists())
print("Behavior file exists:", behav_path.exists())

Spike file exists: True
Behavior file exists: True


In [45]:
with h5py.File(spike_path, "r") as f:

    spikes = f["spikes"]

    yfi_spike_matrix = csc_matrix(
        (
            spikes["data"][:],
            spikes["ir"][:],
            spikes["jc"][:]
        ),
        shape=(
            int(spikes.attrs["MATLAB_sparse"]),
            len(spikes["jc"]) - 1
        )
    )

    regions_obj = f["regionsVect"]

    yfi_region_labels = [
        decode_matlab_string(f, ref)
        for ref in regions_obj[0]
    ]

yfi_behav = pd.read_csv(behav_path)

print("YFI spike matrix:", yfi_spike_matrix.shape)
print("YFI behavior:", yfi_behav.shape)
print("Region labels:", len(yfi_region_labels))

YFI spike matrix: (38, 944701)
YFI behavior: (100, 22)
Region labels: 38


In [46]:
unique_regions, counts = np.unique(
    yfi_region_labels,
    return_counts=True
)

for region, count in zip(unique_regions, counts):
    print(region, count)

acc 9
ent 8
hpc 21


In [47]:
print(yfi_behav.columns.tolist())

display(yfi_behav.head())

['trial', 'tCue1', 'tCue2', 'tCue3', 'presentationEnd', 'tPress1', 'tPress2', 'tPress3', 'keyPress1', 'keyPress2', 'keyPress3', 'choiceEnd', 'timeFeedback', 'cue1', 'cue2', 'operation', 'operationFirst', 'correctAnsw', 'givenAnsw', 'correct', 'correctFb', 'toExclude']


,trial,tCue1,tCue2,tCue3,presentationEnd,tPress1,tPress2,tPress3,keyPress1,keyPress2,...,timeFeedback,cue1,cue2,operation,operationFirst,correctAnsw,givenAnsw,correct,correctFb,toExclude
0,1,43119.166667,43853.200000,44603.933333,45388.000000,49959.066667,50476.233333,NaN,1,1.0,...,55581.133333,3,8,+,1,11,11.0,1,1,0
1,2,56064.966667,56815.666667,57566.366667,58317.100000,61370.033333,NaN,NaN,0,NaN,...,62954.866667,7,7,-,1,0,0.0,1,1,0
2,3,63455.366667,64206.100000,64956.800000,65707.533333,68109.833333,68593.633333,84859.266667,1,1.0,...,86360.700000,3,14,-,1,-11,NaN,0,1,0
3,4,86861.200000,87611.900000,88362.633333,89113.333333,92650.066667,93284.033333,NaN,2,7.0,...,94351.700000,13,14,+,0,27,27.0,1,1,0
4,5,94852.200000,95602.900000,96353.633333,97104.366667,98505.700000,99256.433333,NaN,1,8.0,...,100357.466667,10,8,+,1,18,18.0,1,1,0


In [48]:
print(yfi_behav.columns.tolist())

['trial', 'tCue1', 'tCue2', 'tCue3', 'presentationEnd', 'tPress1', 'tPress2', 'tPress3', 'keyPress1', 'keyPress2', 'keyPress3', 'choiceEnd', 'timeFeedback', 'cue1', 'cue2', 'operation', 'operationFirst', 'correctAnsw', 'givenAnsw', 'correct', 'correctFb', 'toExclude']


In [49]:
yfi_aligned = yfi_behav.copy()

yfi_aligned["operand1_time"] = np.where(
    yfi_aligned["operationFirst"] == 1,
    yfi_aligned["tCue2"],
    yfi_aligned["tCue1"]
)

yfi_aligned["operation_time"] = np.where(
    yfi_aligned["operationFirst"] == 1,
    yfi_aligned["tCue1"],
    yfi_aligned["tCue3"]
)

yfi_aligned["operand2_time"] = np.where(
    yfi_aligned["operationFirst"] == 1,
    yfi_aligned["tCue3"],
    yfi_aligned["tCue2"]
)

In [50]:
presentation_rows = []

for trial_idx, row in yfi_aligned.iterrows():

    # Operand 1
    if (
        pd.notna(row["cue1"])
        and pd.notna(row["operand1_time"])
        and 1 <= row["cue1"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 1,
            "number": int(row["cue1"]),
            "onset_ms": row["operand1_time"]
        })

    # Operand 2
    if (
        pd.notna(row["cue2"])
        and pd.notna(row["operand2_time"])
        and 1 <= row["cue2"] <= 9
    ):
        presentation_rows.append({
            "trial_index": trial_idx,
            "operand": 2,
            "number": int(row["cue2"]),
            "onset_ms": row["operand2_time"]
        })

yfi_presentations = pd.DataFrame(presentation_rows)

In [51]:
print("YFI presentations:", yfi_presentations.shape)

display(
    yfi_presentations.groupby(
        ["number", "operand"]
    ).size().unstack(fill_value=0)
)

print(
    "\nTotal presentations:",
    len(yfi_presentations)
)

YFI presentations: (104, 4)


operand,1,2
number,,
1,7,3
2,4,7
3,6,5
4,7,4
5,5,4
6,6,4
7,5,5
8,11,12
9,4,5



Total presentations: 104


In [52]:
test_neuron = 9

print("Region:", yfi_region_labels[test_neuron])

Region: acc


In [53]:
yfi_test = decode_fr_one_neuron(
    spike_matrix=yfi_spike_matrix,
    presentations=yfi_presentations,
    neuron_id=test_neuron,
    n_shuffles=200,
    random_state=42
)

yfi_test

{'neuron_id': 9,
 'n_presentations': 104,
 'n_folds': np.int64(9),
 'fr_accuracy': 0.07692307692307693,
 'shuffle_mean_accuracy': np.float64(0.11014423076923077),
 'n_extreme': 171,
 'p_value': np.float64(0.8557213930348259),
 'fr_coding': False}

In [54]:
yfi_fr_results = []

n_neurons = yfi_spike_matrix.shape[0]

for neuron_id in range(n_neurons):

    result = decode_fr_one_neuron(
        spike_matrix=yfi_spike_matrix,
        presentations=yfi_presentations,
        neuron_id=neuron_id,
        n_shuffles=200,
        random_state=42
    )

    result["subject"] = "YFI"
    result["region"] = yfi_region_labels[neuron_id]

    yfi_fr_results.append(result)

    print(
        f"YFI: {neuron_id + 1}/{n_neurons}",
        end="\r"
    )

print("\nYFI complete.")

YFI: 38/38
YFI complete.


In [55]:
yfi_fr_df = pd.DataFrame(yfi_fr_results)

print("All recorded units:", len(yfi_fr_df))
print("All FR-coding units:", yfi_fr_df["fr_coding"].sum())

All recorded units: 38
All FR-coding units: 3


In [56]:
mtl_regions = ["hpc", "ent", "amy", "phc"]

yfi_mtl_fr_df = yfi_fr_df[
    yfi_fr_df["region"].isin(mtl_regions)
].copy()

print("YFI MTL neurons:", len(yfi_mtl_fr_df))
print(
    "YFI MTL FR-coding neurons:",
    yfi_mtl_fr_df["fr_coding"].sum()
)
print(
    "YFI MTL FR-coding %:",
    100 * yfi_mtl_fr_df["fr_coding"].mean()
)

YFI MTL neurons: 29
YFI MTL FR-coding neurons: 3
YFI MTL FR-coding %: 10.344827586206897


In [57]:
subjects = [
    "YFF", "YFI", "YFJ", "YFK", "YFL",
    "YFM", "YFP", "YFR", "YFS", "YFT", "YFU"
]

def load_subject_basic(subject):

    subject_dir = raw_dir / subject / "arithmetic"

    if (subject_dir / "spikes.mat").exists():
        spike_path = subject_dir / "spikes.mat"
        spike_key = "spikes"

    elif (subject_dir / "spikesArithmetic.mat").exists():
        spike_path = subject_dir / "spikesArithmetic.mat"
        spike_key = "spikesArithmetic"

    else:
        raise FileNotFoundError(
            f"No arithmetic spike file found for {subject}"
        )

    behav_path = subject_dir / "photoBehavEvents.csv"

    with h5py.File(spike_path, "r") as f:

        spikes = f[spike_key]

        spike_matrix = csc_matrix(
            (
                spikes["data"][:],
                spikes["ir"][:],
                spikes["jc"][:]
            ),
            shape=(
                int(spikes.attrs["MATLAB_sparse"]),
                len(spikes["jc"]) - 1
            )
        )

        regions_obj = f["regionsVect"]

        region_labels = [
            decode_matlab_string(f, ref)
            for ref in regions_obj[0]
        ]

    behav = pd.read_csv(behav_path)

    return spike_matrix, region_labels, behav

In [58]:
subject_data = {}

for subject in subjects:

    print(f"Loading {subject}...")

    spike_matrix, region_labels, behav = (
        load_subject_basic(subject)
    )

    subject_data[subject] = {
        "spikes": spike_matrix,
        "regions": region_labels,
        "behavior": behav
    }

print("\nAll subjects loaded.")

Loading YFF...
Loading YFI...
Loading YFJ...
Loading YFK...
Loading YFL...
Loading YFM...
Loading YFP...
Loading YFR...
Loading YFS...
Loading YFT...
Loading YFU...

All subjects loaded.


In [62]:
mtl_regions = ["hpc", "ent", "amy", "para-hpc"]

master_rows = []

for subject in subjects:

    regions = subject_data[subject]["regions"]

    for neuron_id, region in enumerate(regions):

        if region in mtl_regions:

            master_rows.append({
                "subject": subject,
                "neuron_id": neuron_id,
                "region": region
            })

mtl_neurons = pd.DataFrame(master_rows)

In [63]:
print(
    "Total MTL neurons:",
    len(mtl_neurons)
)

print("\nMTL neurons by region:")

print(
    mtl_neurons["region"]
    .value_counts()
    .sort_index()
)

print("\nMTL neurons by subject:")

print(
    mtl_neurons["subject"]
    .value_counts()
    .sort_index()
)

Total MTL neurons: 554

MTL neurons by region:
region
amy          77
ent          70
hpc         389
para-hpc     18
Name: count, dtype: int64

MTL neurons by subject:
subject
YFF    37
YFI    29
YFJ    45
YFK    44
YFL    57
YFM    61
YFP    43
YFR    65
YFS    59
YFT    52
YFU    62
Name: count, dtype: int64


In [64]:
for subject in ["YFR", "YFS"]:

    behav = subject_data[subject]["behavior"]

    print(f"\n===== {subject} =====")
    print("Shape:", behav.shape)

    display(
        behav[
            [
                "cue1",
                "operation",
                "cue2",
                "operationFirst",
                "tCue1",
                "tCue2",
                "tCue3"
            ]
        ].head(10)
    )


===== YFR =====
Shape: (160, 22)


,cue1,operation,cue2,operationFirst,tCue1,tCue2,tCue3
0,9,+,1,0,21758.966667,22492.266667,23225.600000
1,10,+,5,0,85674.900000,86424.866667,87174.866667
2,10,+,7,0,103741.333333,104491.333333,105241.333333
3,1,+,9,0,114491.200000,115241.200000,115991.200000
4,5,+,4,0,122591.100000,123341.100000,124091.100000
5,10,+,3,0,135574.300000,136324.300000,137074.266667
6,1,+,1,0,145907.500000,146657.500000,147407.500000
7,6,+,5,0,153074.066667,153824.066667,154574.066667
8,6,+,0,0,161024.000000,161773.966667,162523.966667
9,1,+,8,0,170423.866667,171173.866667,171923.866667



===== YFS =====
Shape: (160, 22)


,cue1,operation,cue2,operationFirst,tCue1,tCue2,tCue3
0,6,+,1,0,16352.200000,17102.166667,17852.166667
1,3,+,3,0,29485.333333,30235.300000,30985.300000
2,0,+,0,0,33935.233333,34685.233333,35435.233333
3,7,+,9,0,39018.500000,39768.466667,40518.466667
4,8,+,10,0,46751.700000,47501.700000,48251.666667
5,5,+,6,0,51801.633333,52551.600000,53301.600000
6,7,+,0,0,56634.866667,57384.866667,58134.833333
7,2,+,8,0,61334.800000,62084.766667,62834.766667
8,7,+,7,0,66251.400000,67001.366667,67751.366667
9,8,+,5,0,71001.300000,71751.300000,72501.266667


In [65]:
def build_arithmetic_presentations(subject, behav):

    aligned = behav.copy()

    # -------------------------------------------------
    # Determine operand onset times
    # -------------------------------------------------

    if subject in ["YFR", "YFS"]:

        # Special subjects:
        # operation sign was always Cue2
        aligned["operand1_time"] = aligned["tCue1"]
        aligned["operand2_time"] = aligned["tCue3"]

    else:

        aligned["operand1_time"] = np.where(
            aligned["operationFirst"] == 1,
            aligned["tCue2"],
            aligned["tCue1"]
        )

        aligned["operand2_time"] = np.where(
            aligned["operationFirst"] == 1,
            aligned["tCue3"],
            aligned["tCue2"]
        )

    # -------------------------------------------------
    # Create pooled operand presentations
    # -------------------------------------------------

    rows = []

    for trial_idx, row in aligned.iterrows():

        # Operand 1
        if (
            pd.notna(row["cue1"])
            and pd.notna(row["operand1_time"])
            and 1 <= row["cue1"] <= 9
        ):

            rows.append({
                "trial_index": trial_idx,
                "operand": 1,
                "number": int(row["cue1"]),
                "onset_ms": row["operand1_time"]
            })

        # Operand 2
        if (
            pd.notna(row["cue2"])
            and pd.notna(row["operand2_time"])
            and 1 <= row["cue2"] <= 9
        ):

            rows.append({
                "trial_index": trial_idx,
                "operand": 2,
                "number": int(row["cue2"]),
                "onset_ms": row["operand2_time"]
            })

    return pd.DataFrame(rows)

In [66]:
subject_presentations = {}

for subject in subjects:

    behav = subject_data[subject]["behavior"]

    presentations = build_arithmetic_presentations(
        subject,
        behav
    )

    subject_presentations[subject] = presentations

In [67]:
presentation_summary = []

for subject in subjects:

    p = subject_presentations[subject]

    counts = p["number"].value_counts()

    presentation_summary.append({
        "subject": subject,
        "n_presentations": len(p),
        "min_class_count": counts.min(),
        "max_class_count": counts.max(),
        "numbers_present": len(counts)
    })

presentation_summary = pd.DataFrame(
    presentation_summary
)

display(presentation_summary)

,subject,n_presentations,min_class_count,max_class_count,numbers_present
0,YFF,109,7,17,9
1,YFI,104,9,23,9
2,YFJ,114,4,19,9
3,YFK,114,4,19,9
4,YFL,114,4,19,9
5,YFM,114,4,19,9
6,YFP,158,9,28,9
7,YFR,266,20,36,9
8,YFS,264,24,33,9
9,YFT,477,43,62,9


In [68]:
all_fr_results = []

n_total = len(mtl_neurons)

for k, row in mtl_neurons.iterrows():

    subject = row["subject"]
    neuron_id = int(row["neuron_id"])
    region = row["region"]

    result = decode_fr_one_neuron(
        spike_matrix=subject_data[subject]["spikes"],
        presentations=subject_presentations[subject],
        neuron_id=neuron_id,
        n_shuffles=200,
        random_state=42
    )

    result["subject"] = subject
    result["region"] = region
    result["paper_unit"] = neuron_id + 1

    all_fr_results.append(result)

    print(
        f"{len(all_fr_results)}/{n_total} | "
        f"{subject} | {region} | unit {neuron_id + 1}",
        end="\r"
    )

print("\nAll 554 MTL neurons complete.")

IndexError: index 0 is out of bounds for axis 0 with size 0

Fix bugs 

In [69]:
def cv_fr_accuracy(X, y, random_state=42):

    classes, counts = np.unique(
        y,
        return_counts=True
    )

    n_folds = min(
        10,
        int(counts.min())
    )

    if n_folds < 2:
        return np.nan, n_folds

    cv = StratifiedKFold(
        n_splits=n_folds,
        shuffle=True,
        random_state=random_state
    )

    priors = np.ones(
        len(classes)
    ) / len(classes)

    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in cv.split(X, y):

        X_train = X[train_idx]
        X_test = X[test_idx]

        y_train = y[train_idx]
        y_test = y[test_idx]

        # No usable firing-rate variation
        if np.ptp(X_train[:, 0]) == 0:
            return np.nan, n_folds

        lda = LinearDiscriminantAnalysis(
            priors=priors
        )

        try:
            lda.fit(X_train, y_train)
            y_pred = lda.predict(X_test)

        except (ValueError, IndexError, np.linalg.LinAlgError):
            return np.nan, n_folds

        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)

    accuracy = accuracy_score(
        y_true_all,
        y_pred_all
    )

    return accuracy, n_folds

In [70]:
def decode_fr_one_neuron(
    spike_matrix,
    presentations,
    neuron_id,
    n_shuffles=200,
    random_state=42
):

    window_start_ms = 50
    window_end_ms = 950
    window_sec = 0.9

    X_rows = []
    labels = []

    # ============================================
    # Build firing-rate feature
    # ============================================
    for _, row in presentations.iterrows():

        number = int(row["number"])
        onset = row["onset_ms"]

        start = int(round(
            onset + window_start_ms
        ))

        end = int(round(
            onset + window_end_ms
        ))

        spike_count = spike_matrix[
            neuron_id,
            start:end
        ].sum()

        firing_rate = (
            float(spike_count) / window_sec
        )

        X_rows.append([firing_rate])
        labels.append(number)

    X = np.asarray(X_rows, dtype=float)
    y = np.asarray(labels)

    # ============================================
    # Observed decoding
    # ============================================
    observed_accuracy, n_folds = cv_fr_accuracy(
        X,
        y,
        random_state=random_state
    )

    # Neuron cannot be decoded
    if not np.isfinite(observed_accuracy):

        return {
            "neuron_id": neuron_id,
            "n_presentations": len(y),
            "n_folds": n_folds,
            "fr_accuracy": np.nan,
            "shuffle_mean_accuracy": np.nan,
            "n_extreme": np.nan,
            "n_valid_shuffles": 0,
            "p_value": np.nan,
            "fr_coding": False
        }

    # ============================================
    # Permutation test
    # ============================================
    rng = np.random.default_rng(
        random_state
    )

    shuffled_accuracies = []

    for _ in range(n_shuffles):

        y_shuffled = rng.permutation(y)

        shuffled_accuracy, _ = cv_fr_accuracy(
            X,
            y_shuffled,
            random_state=random_state
        )

        if np.isfinite(shuffled_accuracy):
            shuffled_accuracies.append(
                shuffled_accuracy
            )

    shuffled_accuracies = np.asarray(
        shuffled_accuracies,
        dtype=float
    )

    # No valid permutations
    if len(shuffled_accuracies) == 0:

        return {
            "neuron_id": neuron_id,
            "n_presentations": len(y),
            "n_folds": n_folds,
            "fr_accuracy": observed_accuracy,
            "shuffle_mean_accuracy": np.nan,
            "n_extreme": np.nan,
            "n_valid_shuffles": 0,
            "p_value": np.nan,
            "fr_coding": False
        }

    n_extreme = np.sum(
        shuffled_accuracies >= observed_accuracy
    )

    p_value = (
        1 + n_extreme
    ) / (
        1 + len(shuffled_accuracies)
    )

    return {
        "neuron_id": neuron_id,
        "n_presentations": len(y),
        "n_folds": n_folds,
        "fr_accuracy": observed_accuracy,
        "shuffle_mean_accuracy":
            shuffled_accuracies.mean(),
        "n_extreme": int(n_extreme),
        "n_valid_shuffles":
            len(shuffled_accuracies),
        "p_value": p_value,
        "fr_coding":
            bool(p_value < 0.05)
    }

In [71]:
print(
    "Completed before crash:",
    len(all_fr_results)
)

Completed before crash: 114


In [72]:
start_idx = len(all_fr_results)

print(
    f"Resuming from {start_idx + 1}/554"
)

for pos in range(
    start_idx,
    len(mtl_neurons)
):

    row = mtl_neurons.iloc[pos]

    subject = row["subject"]
    neuron_id = int(row["neuron_id"])
    region = row["region"]

    result = decode_fr_one_neuron(
        spike_matrix=subject_data[subject]["spikes"],
        presentations=subject_presentations[subject],
        neuron_id=neuron_id,
        n_shuffles=200,
        random_state=42
    )

    result["subject"] = subject
    result["region"] = region
    result["paper_unit"] = neuron_id + 1

    all_fr_results.append(result)

    print(
        f"{len(all_fr_results)}/554 | "
        f"{subject} | {region} | "
        f"unit {neuron_id + 1}",
        end="\r"
    )

print("\nAll 554 MTL neurons complete.")

Resuming from 115/554
554/554 | YFU | hpc | unit 70it 68
All 554 MTL neurons complete.


In [74]:
all_fr_df = pd.DataFrame(all_fr_results)

print("Rows:", len(all_fr_df))

print(
    "NaN FR accuracies:",
    all_fr_df["fr_accuracy"].isna().sum()
)

print(
    "NaN p-values:",
    all_fr_df["p_value"].isna().sum()
)

print(
    "FR-coding neurons:",
    all_fr_df["fr_coding"].sum()
)

print(
    "Pooled FR-coding percentage:",
    100 * all_fr_df["fr_coding"].mean()
)

Rows: 554
NaN FR accuracies: 5
NaN p-values: 5
FR-coding neurons: 17
Pooled FR-coding percentage: 3.068592057761733


17/554

In [77]:
17/554

0.030685920577617327

In [75]:
nan_fr = all_fr_df[
    all_fr_df["fr_accuracy"].isna()
][
    [
        "subject",
        "neuron_id",
        "paper_unit",
        "region",
        "n_presentations",
        "fr_accuracy",
        "p_value",
        "fr_coding"
    ]
]

display(nan_fr)

,subject,neuron_id,paper_unit,region,n_presentations,fr_accuracy,p_value,fr_coding
163,YFL,27,28,hpc,114,NaN,NaN,False
242,YFM,63,64,hpc,114,NaN,NaN,False
285,YFP,41,42,hpc,158,NaN,NaN,False
300,YFP,82,83,hpc,158,NaN,NaN,False
330,YFR,14,15,hpc,266,NaN,NaN,False


In [76]:
all_fr_df.to_csv(
    tables_dir / "figure1_arithmetic_FR_all554.csv",
    index=False
)

print("Saved.")

Saved.


In [78]:
fr_subject_summary = (
    all_fr_df
    .groupby("subject")
    .agg(
        n_neurons=("neuron_id", "size"),
        n_valid=("fr_accuracy", "count"),
        n_fr_coding=("fr_coding", "sum")
    )
    .reset_index()
)

# Use total MTL neurons as denominator
fr_subject_summary["fr_coding_percent"] = (
    100
    * fr_subject_summary["n_fr_coding"]
    / fr_subject_summary["n_neurons"]
)

display(fr_subject_summary)

,subject,n_neurons,n_valid,n_fr_coding,fr_coding_percent
0,YFF,37,37,1,2.702703
1,YFI,29,29,3,10.344828
2,YFJ,45,45,2,4.444444
3,YFK,44,44,0,0.000000
4,YFL,57,56,0,0.000000
5,YFM,61,60,3,4.918033
6,YFP,43,41,1,2.325581
7,YFR,65,64,0,0.000000
8,YFS,59,59,1,1.694915
9,YFT,52,52,2,3.846154


In [79]:
from scipy.stats import sem

mean_fr = fr_subject_summary[
    "fr_coding_percent"
].mean()

sem_fr = sem(
    fr_subject_summary["fr_coding_percent"]
)

print(
    f"Subject-level mean FR coding: "
    f"{mean_fr:.2f}%"
)

print(
    f"Subject-level SEM: "
    f"{sem_fr:.2f}%"
)

Subject-level mean FR coding: 3.34%
Subject-level SEM: 0.96%


In [80]:
fr_subject_summary.to_csv(
    tables_dir / "figure1M_arithmetic_FR_subject_summary.csv",
    index=False
)